# Arabic Legal AI Assistant — Hybrid RAG with Legal-Index Routing

This notebook builds a Retrieval-Augmented Generation (RAG) assistant that answers
questions about **Egyptian law** using only the text of the uploaded legal PDFs.

Knowledge sources used in this notebook:
- `Family_Law.pdf` — قانون الأحوال الشخصية (Personal Status Law)
- `Labor_Law.pdf` — قانون رقم 14 لسنة 2025 بإصدار قانون العمل (Labor Law)
## Architecture

```
User Query (Arabic)
        │
        ▼
 Legal Index Routing  ──►  which law? which chapter? which article range?
        │
        ▼
 Scoped Hybrid Retrieval  (inside that law/section only)
        │
   ┌────┴─────┐
   │          │
BM25       FAISS (dense embeddings)
   │          │
   └────┬─────┘
        ▼
   Merge candidates
        ▼
  Cross-Encoder Re-ranker
        ▼
   Top-K passages
        ▼
   LLM  +  Conversation Memory
        ▼
 Grounded Arabic answer (Law, Article, Explanation, Source)
```


# **Install dependencies**

In [1]:
!pip install -q \
    "langchain-core==0.3.75" \
    "langchain==0.3.27" \
    "langchain-community==0.3.27" \
    "langchain-text-splitters==0.3.9" \
    "langchain-huggingface==0.3.1" \
    "bitsandbytes>=0.46.1" \
    pymupdf sentence-transformers rank_bm25 faiss-cpu \
    transformers accelerate gspread gspread-dataframe openpyxl pandas peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.0/444.0 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.3 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 74.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 62.1 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.75 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.75 which is incompatible.


# **Imports & global configuration**

In [4]:
import os, re, glob
import pandas as pd
import numpy as np
import fitz  # PyMuPDF
from dataclasses import dataclass, field
from typing import List, Dict, Optional

from langchain_core.documents import Document
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ---- Paths -------------------------------------------------------
LAW_FILES = {
    "قانون الأحوال الشخصية": "/kaggle/input/datasets/reemadel234/legal-index-data/Family_Law.pdf",
    "قانون العمل":            "/kaggle/input/datasets/reemadel234/legal-index-data/Labor_Law.pdf",
}
# ---- Models --------------------------------------------------------------
EMBEDDING_MODEL_NAME = "BAAI/bge-m3"                 # dense embeddings (Arabic-capable, multilingual)
RERANKER_MODEL_NAME  = "BAAI/bge-reranker-v2-m3"      # cross-encoder re-ranker (Arabic-capable, multilingual)
LLM_MODEL_NAME       = "Qwen/Qwen2.5-7B-Instruct"     # generation model (strong Arabic + follows grounding instructions well)


TOP_K_FINAL   = 4     # passages handed to the LLM after re-ranking
TOP_K_HYBRID  = 10    #BM25+FAISS before re-ranking


## Step 1 — Load the PDFs and split them into **article-level** chunks

Egyptian legal PDFs are structured around the word **"مادة" (Article)**. Splitting
on that marker (rather than a generic character/token splitter) keeps each chunk
aligned with a citable article — which matters because every answer must report
an article number.

`PyMuPDF (fitz)` is used instead of `PyPDFLoader`/`pdfplumber` because it preserves
correct Arabic reading order for these two files (verified against the raw PDFs);
`pdfplumber` returned visually-reversed glyphs on this specific PDF export.


In [5]:
import unicodedata

DIGIT_TRANS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")

# Real numbered headers in both PDFs are written "مادة N" (sometimes with
# spaces inside the word, or RTL-flipped parens). "المادة N" only appears
# in inline cross-references, so the ال prefix is deliberately NOT matched.
ARTICLE_PATTERN = re.compile(
    r"(?:^|\n)[\s\)\(]*م\s*ا\s*د\s*[ةه][\s\)\(:]*([0-9٠-٩۰-۹]+)"
    r"[\s\)\(]*(مكرر[اً]*(?:\s*(?:ثانيا|ثالثا|رابعا))?)?",
    re.MULTILINE)

REVERSE_DIGITS = {
    "قانون العمل": True,
    "قانون الأحوال الشخصية": False,
}

# The family PDF is a compilation of several laws, each restarting at مادة 1.
# The repair below detects each restart as a new "part"; name them here so
# citations can say WHICH law inside the compilation an article belongs to.
PART_NAMES = {
    "قانون الأحوال الشخصية": {
        1: "قانون رقم 25 لسنة 1920 (النفقة)",
        2: "قانون رقم 25 لسنة 1929 (أحكام الأحوال الشخصية)",
        3: "قانون رقم 1 لسنة 2000 (إجراءات التقاضي)",
        4: "قرار وزير العدل 1086 لسنة 2000",
        5: "قرار وزير العدل 1087 لسنة 2000 (الرؤية)",
        6: "قرار وزير العدل 1088 لسنة 2000 (الجرد)",
        7: "قرار وزير العدل 1089 لسنة 2000 (الأخصائيون)",
        8: "قرار وزير العدل 1090 لسنة 2000 (السجل)",
        9: "قانون رقم 10 لسنة 2004 (محاكم الأسرة)",
    }
}

def fix_reversed_digits(text: str) -> str:
    return re.sub(r"[٠-٩]+", lambda m: m.group(0)[::-1], text)

def extract_pdf_text(path: str, reverse_digits: bool = False) -> str:
    doc = fitz.open(path)
    pages = [page.get_text() for page in doc]
    doc.close()
    text = "\n".join(pages)
    text = unicodedata.normalize("NFKC", text)  # ﺍﳌﺎﺩﺓ -> المادة
    if reverse_digits:
        text = fix_reversed_digits(text)
    return text

def _candidates(tok: str) -> set:
    """All plausible original readings of a possibly-corrupted number."""
    opts = [""]
    for ch in tok:
        subs = ["1", "9", "0"] if ch == "1" else [ch]
        opts = [p + s for p in opts for s in subs]
    opts += [o[::-1] for o in opts]              # reversed-run variants
    return {int(o) for o in opts if o and not o.startswith("0")}

def repair_article_numbers(matches) -> list:
    """For each regex match, return (article_label, part_number) with the
    true article number recovered from sequence context."""
    out, prev, part = [], 0, 1
    for m in matches:
        tok = m.group(1).translate(DIGIT_TRANS)
        suffix = (m.group(2) or "").strip()
        cands = _candidates(tok)
        if suffix and prev in cands:
            num = prev                            # مكرر repeats base number
        elif prev + 1 in cands:
            num = prev + 1                        # normal increment
        elif prev + 2 in cands:
            num = prev + 2                        # tolerate one gap
        elif 1 in cands and prev >= 2:
            num, part = 1, part + 1               # restart => new sub-law
        elif prev in cands:
            num = prev                            # duplicated header
        else:
            num = min(cands)
        prev = num
        out.append((str(num) + ((" " + suffix) if suffix else ""), part))
    return out

def split_into_articles(law_name: str, full_text: str) -> List[Document]:
    matches = list(ARTICLE_PATTERN.finditer(full_text))
    docs = []
    if not matches:
        return [Document(page_content=full_text,
                         metadata={"law_name": law_name, "article": None,
                                   "source": law_name})]
    if matches[0].start() > 0:
        preamble = full_text[: matches[0].start()].strip()
        if len(preamble) > 30:
            docs.append(Document(page_content=preamble,
                                 metadata={"law_name": law_name,
                                           "article": "مقدمة",
                                           "source": law_name}))
    labels = repair_article_numbers(matches)
    part_names = PART_NAMES.get(law_name, {})
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)
        chunk_text = full_text[start:end].strip()
        if len(chunk_text) < 5:
            continue
        article_no, part = labels[i]
        sub_law = part_names.get(part, "")
        # Prepend the CORRECTED header so keyword search matches the true
        # article number even though the raw text carries the corrupted one.
        header = f"[المادة {article_no}"
        header += f" — {sub_law}]" if sub_law else "]"
        docs.append(Document(
            page_content=header + "\n" + chunk_text,
            metadata={"law_name": law_name, "article": article_no,
                      "sub_law": sub_law, "source": law_name},
        ))
    return docs

def load_all_laws(law_files: Dict[str, str]) -> Dict[str, List[Document]]:
    law_docs = {}
    for law_name, path in law_files.items():
        if not os.path.exists(path):
            print(f"⚠️  Missing file for '{law_name}': {path} — upload it via the Colab Files panel (left sidebar)")
            continue
        text = extract_pdf_text(path, reverse_digits=REVERSE_DIGITS.get(law_name, False))
        articles = split_into_articles(law_name, text)
        law_docs[law_name] = articles
        print(f"{law_name}: {len(articles)} article-level chunks extracted from {path}")
    return law_docs

law_documents = load_all_laws(LAW_FILES)

قانون الأحوال الشخصية: 189 article-level chunks extracted from /kaggle/input/datasets/reemadel234/legal-index-data/Family_Law.pdf
قانون العمل: 299 article-level chunks extracted from /kaggle/input/datasets/reemadel234/legal-index-data/Labor_Law.pdf


## Step 2 — The Legal Index (routing layer)

The Legal Index is **metadata only** — it never contains legal text, only enough
information to point the retriever at the right law and article range. This is
what lets the assistant avoid searching every PDF for every question.

| Column | Purpose |
|---|---|
| `main_topic` | Broad legal domain (e.g. "الأحوال الشخصية") |
| `subtopic` | Specific matter (e.g. "الخلع", "إنهاء عقد العمل") |
| `law_name` | Must match a key in `LAW_FILES` |
| `chapter` | Chapter/section title as it appears in the law |
| `article_range` | e.g. `"20-20"` or `"110-120"` |
| `keywords` | Arabic keywords/synonyms used for matching the user's query |
| `description` | One-line description used for semantic matching |



In [6]:
# ------------------------------------------------------------------
# Legal Index from an uploaded CSV file (indexsheet.csv)
# Accepts the user's sheet format:
#   Main_Topic | Subtopic | Law_Name | Part | Start_Article | End_Article | Keywords | Description
# and maps it to the pipeline's internal column names.
# ------------------------------------------------------------------

INDEX_CSV_PATH = "/kaggle/input/datasets/reemadel234/legal-index-data/Indexsheet.csv"

COLUMN_MAP = {           # your CSV header  ->  pipeline column
    "Main_Topic":    "main_topic",
    "Subtopic":      "subtopic",
    "Law_Name":      "law_name",
    "Part":          "chapter",
    "Keywords":      "keywords",
    "Description":   "description",
}

REQUIRED_COLS = ["main_topic", "subtopic", "law_name", "chapter",
                 "article_range", "keywords", "description"]

def load_legal_index_from_csv(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"'{path}' not found — upload it via the Files panel (left sidebar).")

    df = pd.read_csv(path, encoding="utf-8-sig").dropna(how="all")
    df.columns = [str(c).strip() for c in df.columns]

    # Rename the user's headers to the pipeline's names
    df = df.rename(columns=COLUMN_MAP)

    def normalize_law_name(name: str) -> str:
        return "قانون العمل" if "عمل" in str(name) else "قانون الأحوال الشخصية"

    df["law_name_original"] = df["law_name"]          # keep the precise name for display
    df["law_name"] = df["law_name"].map(normalize_law_name)


    # Build article_range ("20-20") from Start_Article / End_Article
    if "Start_Article" in df.columns and "End_Article" in df.columns:
        start = df["Start_Article"].astype(str).str.strip()
        end = df["End_Article"].astype(str).str.strip()
        end = end.where((end != "") & (end.str.lower() != "nan"), start)
        df["article_range"] = start + "-" + end
        df = df.drop(columns=["Start_Article", "End_Article"])

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}. "
                         f"Found columns: {list(df.columns)}")

    df = df[df["law_name"].astype(str).str.strip() != ""].copy()
    for c in REQUIRED_COLS:
        df[c] = df[c].astype(str).str.strip()

    unknown = set(df["law_name"]) - set(LAW_FILES.keys())
    if unknown:
        print(f"⚠️  law_name values not matching LAW_FILES (routing will skip these): {unknown}")
        print(f"    Expected exactly: {list(LAW_FILES.keys())}")

    df["index_id"] = range(len(df))
    print(f"Loaded {len(df)} Legal Index rows from {path}")
    return df

legal_index_df = load_legal_index_from_csv(INDEX_CSV_PATH)
legal_index_df.head()

Loaded 81 Legal Index rows from /kaggle/input/datasets/reemadel234/legal-index-data/Indexsheet.csv


,main_topic,subtopic,law_name,chapter,keywords,description,law_name_original,article_range,index_id
0,أحكام عامة,نطاق تطبيق القانون,قانون الأحوال الشخصية,الباب الأول,أحكام عامة، تعريفات، إجراءات,يتناول الأحكام العامة التي تحكم تطبيق القانون ...,قانون تنظيم بعض أوضاع وإجراءات التقاضي في مسائ...,1-5,0
1,اختصاص المحاكم,الاختصاص النوعي,قانون الأحوال الشخصية,الباب الثاني,اختصاص المحكمة، الولاية على النفس، الولاية على...,يحدد اختصاص محاكم الأحوال الشخصية بحسب نوع الن...,قانون تنظيم بعض أوضاع وإجراءات التقاضي في مسائ...,6-14,1
2,ختصاص المحاكم,الاختصاص المحلي,قانون الأحوال الشخصية,الباب الثاني,الاختصاص المحلي، موطن المدعي، موطن المدعى عليه,يوضح المحكمة المختصة مكانيًا بنظر دعاوى الأحوا...,قانون تنظيم بعض أوضاع وإجراءات التقاضي في مسائ...,15-17,2
3,رفع الدعوى,إجراءات رفع الدعوى,قانون الأحوال الشخصية,الباب الثالث,رفع الدعوى، إثبات الزواج، الصلح، الحكمين,يوضح شروط قبول الدعوى وإجراءات نظرها أمام المح...,قانون تنظيم بعض أوضاع وإجراءات التقاضي في مسائ...,17-23,3
4,الطلاق,الصلح بين الزوجين,قانون الأحوال الشخصية,الباب الثالث,صلح، حكمين، إصلاح,ينظم إجراءات الصلح وندب الحكمين قبل الفصل في د...,قانون تنظيم بعض أوضاع وإجراءات التقاضي في مسائ...,18-19,4


## Step 3 — Embedding model & re-ranker: recommendation

Requirements from the brief: strong **Modern Standard Arabic** understanding,
legal-terminology accuracy, and reliable semantic retrieval.

| Purpose | Model | Why |
|---|---|---|
| Dense embeddings | **`BAAI/bge-m3`** | Multilingual (100+ languages incl. Arabic), trained explicitly for retrieval, strong on long/legal-style passages, supports dense+sparse+multi-vector — one model instead of stitching several together. Free, open-source, runs locally. |
| Cross-encoder re-ranker | **`BAAI/bge-reranker-v2-m3`** | Same family as the embedder so relevance scoring is consistent; multilingual and specifically validated on Arabic in public benchmarks; much more accurate than embedding-similarity alone for the final top-k ordering. |
| Alternative embeddings | `intfloat/multilingual-e5-large` | Also strong on Arabic, slightly lighter than bge-m3 if you need less VRAM. |
| Arabic-specific option | `Omartificial-Intelligence-Space/GATE-AraBert-v1` | Purpose-built Arabic sentence embeddings — worth A/B testing against bge-m3 on your own legal queries, but has a narrower training distribution (mostly MSA, less legal text) than bge-m3. |
| Commercial option | OpenAI `text-embedding-3-large` + Cohere `rerank-multilingual-v3` | Good Arabic support, no local GPU needed, but sends legal text to a third-party API — a data-sensitivity trade-off worth flagging for a legal use case. |



In [7]:
# ============================================================
# 4. Load embedding model + build per-law FAISS + BM25 retrievers
# ============================================================

embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_stores: Dict[str, FAISS] = {}
bm25_retrievers: Dict[str, BM25Retriever] = {}

for law_name, docs in law_documents.items():
    if not docs:
        continue
    vector_stores[law_name] = FAISS.from_documents(docs, embedding_model)
    bm25 = BM25Retriever.from_documents(docs)
    bm25.k = TOP_K_HYBRID
    bm25_retrievers[law_name] = bm25
    print(f"Built FAISS + BM25 retrievers for {law_name} ({len(docs)} chunks).")

# Also embed the Legal Index descriptions themselves, for the routing step below.
legal_index_texts = (
    legal_index_df["subtopic"].fillna("") + " — " +
    legal_index_df["description"].fillna("") + " — " +
    legal_index_df["keywords"].fillna("")
).tolist()
legal_index_embeddings = embedding_model.embed_documents(legal_index_texts)
legal_index_embeddings = np.array(legal_index_embeddings)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Built FAISS + BM25 retrievers for قانون الأحوال الشخصية (189 chunks).
Built FAISS + BM25 retrievers for قانون العمل (299 chunks).


## Step 4 — Legal Index routing: pick the law *before* retrieving text

`route_query()` embeds the user's question and compares it against the Legal
Index rows (not the legal text itself) to choose the most likely `law_name` +
`article_range`. This is the "narrow the search space first" step from the brief.
If the top match is weak, the system searches across all laws instead of
guessing — safer than a hard, always-commit routing decision.


In [8]:
ROUTING_CONFIDENCE_THRESHOLD = 0.35  # cosine similarity; tune against your own test queries

def cosine_sim(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a_n = a / (np.linalg.norm(a, axis=-1, keepdims=True) + 1e-8)
    b_n = b / (np.linalg.norm(b, axis=-1, keepdims=True) + 1e-8)
    return a_n @ b_n.T


def route_query(query: str, top_n: int = 2) -> List[dict]:
    """Return the top-N Legal Index rows most relevant to the query, or [] if
    nothing clears the confidence threshold (=> fall back to searching all laws)."""
    q_emb = np.array(embedding_model.embed_query(query)).reshape(1, -1)
    sims = cosine_sim(q_emb, legal_index_embeddings)[0]
    order = np.argsort(-sims)[:top_n]
    matches = []
    for idx in order:
        if sims[idx] >= ROUTING_CONFIDENCE_THRESHOLD:
            row = legal_index_df.iloc[idx].to_dict()
            row["similarity"] = float(sims[idx])
            matches.append(row)
    return matches


def article_in_range(article: Optional[str], article_range: str) -> bool:
    """Loose check used only to *prefer* in-range chunks, never to hard-exclude."""
    if not article:
        return False
    m = re.match(r"(\d+)", str(article))       # "11 مكرر" -> 11
    if not m:
        return False
    art = int(m.group(1))
    nums = re.findall(r"\d+", str(article_range))
    if not nums:
        return False
    lo, hi = int(nums[0]), int(nums[-1])       # handles "20-20", "20", "1-5 مكرر"
    return lo <= art <= hi


## Step 5 — Hybrid retrieval (BM25 + FAISS) and re-ranking

Given the routed law(s), `hybrid_retrieve()`:
1. Runs BM25 and FAISS retrieval **inside the routed law(s) only** via
   LangChain's `EnsembleRetriever` (0.5 / 0.5 weighting — tune per your own tests).
2. Merges and de-duplicates candidates.
3. Re-ranks the merged set with the `bge-reranker-v2-m3` cross-encoder, which
   scores the *(query, passage)* pair directly instead of relying on embedding
   similarity — this is what fixes the "similar wording, wrong article" problem.
4. Returns the top `TOP_K_FINAL` passages.


In [9]:
reranker = CrossEncoder(RERANKER_MODEL_NAME, max_length=512)

def build_ensemble_retriever(law_name: str) -> Optional[EnsembleRetriever]:
    if law_name not in vector_stores:
        return None
    dense = vector_stores[law_name].as_retriever(search_kwargs={"k": TOP_K_HYBRID})
    sparse = bm25_retrievers[law_name]
    return EnsembleRetriever(retrievers=[sparse, dense], weights=[0.5, 0.5])


def hybrid_retrieve(query: str, verbose: bool = False) -> List[Document]:
    routed = route_query(query)

    if routed:
        target_laws = list({r["law_name"] for r in routed})
        if verbose:
            for r in routed:
                print(f"  → routed to: {r['law_name']} / {r['subtopic']} "
                      f"(articles {r['article_range']}, sim={r['similarity']:.2f})")
    else:
        target_laws = list(vector_stores.keys())
        if verbose:
            print("  → routing confidence too low, searching across all laws")

    candidates: List[Document] = []
    seen = set()
    for law_name in target_laws:
        ensemble = build_ensemble_retriever(law_name)
        if ensemble is None:
            continue
        for doc in ensemble.invoke(query):
            key = (doc.metadata.get("law_name"), doc.metadata.get("article"), doc.page_content[:50])
            if key not in seen:
                seen.add(key)
                candidates.append(doc)

    if not candidates:
        return []

    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)

    # Small boost for chunks inside the routed article range(s), so the
    # Legal Index doesn't just pick the law — it also nudges the ranking
    # toward the chapter it identified. Soft preference, never a filter.
    boosted = []
    for doc, score in zip(candidates, scores):
        in_range = any(
            r["law_name"] == doc.metadata.get("law_name")
            and article_in_range(doc.metadata.get("article"), r["article_range"])
            for r in routed
        )
        boosted.append((doc, score + (0.5 if in_range else 0.0)))
    ranked = sorted(boosted, key=lambda x: x[1], reverse=True)
    top_docs = [doc for doc, score in ranked[:TOP_K_FINAL]]

    if verbose:
        print(f"  → {len(candidates)} candidates merged, top {len(top_docs)} kept after re-ranking")
        for doc, score in ranked[:TOP_K_FINAL]:
            print(f"     [{score:+.2f}] المادة {doc.metadata.get('article')} — {doc.metadata.get('sub_law') or doc.metadata.get('law_name')}")
    return top_docs

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

In [10]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(torch.cuda.memory_allocated() / 1e9, "GB allocated")
print(torch.cuda.memory_reserved() / 1e9, "GB reserved")

4.551875072 GB allocated
5.154799616 GB reserved


In [11]:
import gc
import torch

# after you've finished building vector_stores / bm25_retrievers / legal_index_embeddings:
del embedding_model
gc.collect()
torch.cuda.empty_cache()

# reload a lightweight CPU copy for use in route_query() and query-time embedding
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},         # force CPU this time
    encode_kwargs={"normalize_embeddings": True},
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Step 6 — LLM

`Qwen2.5-7B-Instruct` is the default: noticeably better Arabic fluency

In [12]:
# ============================================================
# 7. LLM loading (4-bit quantized so Qwen2.5-7B fits on a T4
#    alongside the embedding model and the re-ranker)
# ============================================================
from transformers import BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)

if torch.cuda.is_available():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        quantization_config=bnb,
        device_map="auto",
    )
else:
    # CPU fallback — only sensible with a small model like Qwen2.5-0.5B/1.5B
    llm_model = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_NAME,
        dtype=torch.float32,
    )

def generate(prompt: str, max_new_tokens: int = 600) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=6000).to(llm_model.device)
    with torch.no_grad():
        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

## Step 7 — Conversational memory + grounded answer generation

Two responsibilities are kept separate on purpose:
1. **`contextualize_query()`** — rewrites a follow-up question
2. **`answer_query()`** — retrieves, then asks the LLM to answer **only** from
   the retrieved passages, in the exact Law / Article / Answer / Source format
   the brief requires, and to return the fixed refusal sentence if the passages
   don't support an answer.


In [13]:

REFUSAL_MESSAGE = "لم أتمكن من العثور على نص قانوني في المستندات المتاحة يدعم الإجابة على هذا السؤال."

SYSTEM_INSTRUCTIONS = """أنت مساعد قانوني متخصص في القانون المصري.
اكتب إجابتك كاملة باللغة العربية الفصحى فقط. لا تستخدم أي لغة أخرى إطلاقاً (لا الإنجليزية ولا الصينية).
أجب فقط بالاعتماد على النصوص القانونية المسترجعة أدناه. لا تخترع أي معلومة قانونية أبداً.
إذا لم تكن النصوص المسترجعة كافية للإجابة، أجب حرفياً بالعبارة التالية فقط:
"{refusal}"

عند الإجابة، التزم بالتنسيق التالي:
القانون: <اسم القانون>
المادة: <رقم المادة إن وجد>
الإجابة: <شرح موجز بالعربية بالاعتماد على النص فقط>
المصدر: <اسم القانون / المادة>
""".format(refusal=REFUSAL_MESSAGE)


class ChatMemory:
    def __init__(self):
        self.turns: List[Dict[str, str]] = []  # [{"role": "user"/"assistant", "content": ...}]

    def add(self, role: str, content: str):
        self.turns.append({"role": role, "content": content})

    def history_text(self, last_n: int = 6, max_chars_per_turn: int = 400) -> str:
        recent = self.turns[-last_n:]
        return "\n".join(f"{t['role']}: {t['content'][:max_chars_per_turn]}" for t in recent)

    def clear(self):
        self.turns = []


memory = ChatMemory()


def looks_like_followup(q: str) -> bool:
    q = q.strip()
    return (len(q) < 25
            or q.startswith(("و", "ثم", "طيب", "وماذا", "ماذا عنه", "اشرحه"))
            or any(w in q for w in ["ذلك", "هذا الحكم", "نفس", "أيضًا", "ايضا"]))

def contextualize_query(query: str) -> str:
    """Rewrite ONLY follow-up-looking questions; self-contained ones pass through."""
    if not memory.turns or not looks_like_followup(query):
        return query
    prompt = f"""سجل المحادثة:
{memory.history_text()}

السؤال الأخير يعتمد على السياق السابق: "{query}"
أعد صياغته ليصبح سؤالاً مستقلاً مفهوماً بدون السياق. أعد السؤال المعاد صياغته فقط في سطر واحد، ولا تُعد أي سؤال سابق من المحادثة.

السؤال المعاد صياغته:"""
    rewritten = generate(prompt, max_new_tokens=60).split("\n")[0].strip()
    # Guard: if the "rewrite" is just an earlier question copied back, discard it
    if any(rewritten in t["content"] or t["content"] in rewritten
           for t in memory.turns if t["role"] == "user"):
        return query
    return rewritten or query


def format_context(docs: List[Document]) -> str:
    blocks = []
    for d in docs:
        law = d.metadata.get("law_name", "غير معروف")
        sub = d.metadata.get("sub_law", "")
        if sub:
            law = f"{law} — {sub}"
        art = d.metadata.get("article", "-")
        blocks.append(f"[القانون: {law} | المادة: {art}]\n{d.page_content[:2500]}")
    return "\n\n---\n\n".join(blocks)


def answer_query(raw_query: str, verbose: bool = False) -> str:
    standalone_query = contextualize_query(raw_query)
    if verbose and standalone_query != raw_query:
        print(f"  (معاد صياغته إلى: {standalone_query})")

    docs = hybrid_retrieve(standalone_query, verbose=verbose)

    if not docs:
        memory.add("user", raw_query)
        memory.add("assistant", REFUSAL_MESSAGE)
        return REFUSAL_MESSAGE

    context = format_context(docs)
    prompt = f"""{SYSTEM_INSTRUCTIONS}

النصوص القانونية المسترجعة:
{context}

سؤال المستخدم: {standalone_query}

الإجابة:"""

    answer = generate(prompt)

    # Keep only Arabic-script output; stop at the first non-Arabic drift
    lines = []
    for line in answer.split("\n"):
        if re.search(r"[\u4e00-\u9fff]", line):   # Chinese characters => drift began
            break
        lines.append(line)
    answer = "\n".join(lines).strip()
    if REFUSAL_MESSAGE in answer:
        answer = REFUSAL_MESSAGE
    memory.add("user", raw_query)
    memory.add("assistant", answer)
    return answer


def ask(query: str, verbose: bool = True) -> None:
    print(f"\n{'─' * 80}")
    print(f"🧑 السؤال: {query}\n")
    answer = answer_query(query, verbose=verbose)
    print(f"\n🤖 {answer}")


## Step 8 — Testing & evaluation

In [14]:
memory.clear()

# 1) Family law, semantic + routing test
ask("ما هي شروط الخلع في القانون المصري؟")

# 2) Follow up question
ask("هل يمكن للمحكمة رفض الدعوى حتى مع توافر الشروط")

# 3) Labor law, different document -> tests routing switches law correctly
ask("ما هي حقوق العامل عند إنهاء عقد العمل؟")

# 4) Exact article-number query -> tests BM25 half of the hybrid retriever
ask("ماذا تنص المادة 20 من قانون الأحوال الشخصية؟")

# 5) Out-of-scope question -> should trigger the refusal sentence, not a hallucinated answer
ask("ما هي عقوبة القتل العمد في قانون العقوبات؟")

# 6) Exact article-number query on the second law
ask("ماذا تنص المادة 1 من قانون العمل؟")



────────────────────────────────────────────────────────────────────────────────
🧑 السؤال: ما هي شروط الخلع في القانون المصري؟

  → routed to: قانون الأحوال الشخصية / الخلع (articles 20-20, sim=0.63)
  → routed to: قانون العمل / إنهاء العقد بسبب إخلال صاحب العمل (articles 70-70, sim=0.46)


The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  → 37 candidates merged, top 4 kept after re-ranking
     [+0.78] المادة 20 — قانون رقم 1 لسنة 2000 (إجراءات التقاضي)
     [+0.04] المادة 6 — قانون العمل
     [+0.02] المادة 168 — قانون العمل
     [+0.02] المادة 114 — قانون العمل

🤖 القانون: قانون الأحوال الشخصية — قانون رقم 1 لسنة 2000 (إجراءات التقاضي)
المادة: 20
الإجابة: الزوجان يمكنهما التراضي على الخلع، فلو تراضيا عليه واقامت الزوجة دعوى بطلب منه وافتبدت نفسها وخالعت زوجها بالتنازل عن جميع حقوقها المالية الشرعية وردت عليه الصداق الذي أعطاه لها، حكمت المحكمة بتطليقها عليه. وفي حالة عدم التراضي، تحكم المحكمة بتطليق الزوجة بدلًا من الخلع إذا ثبتت محاولتها إصلاح الأمر بين الزوجين وتبين أن الزوجة ترغب في إنهاء الحياة الزوجية بسبب البغض وانعدام السبيل لاستمرارها، ويجوز أن يكون ذلك مقابل الخلع بإسقاط حضانة أو نفقة أو أي حق آخر مبني على حق. ويقع الخلع في جميع الأحوال طالما بين، ويكون الحكم غير قابل للطعن عليه بغير طريق من طرق الطعن.
المصدر: قانون الأحوال الشخصية — قانون رقم 1 لسنة 2000 (إجراءات التقاضي) / المادة 20

─────────────────────